**Ερώτημα 1ο**

Δημιουργία 6 dataset, normalisation & check NaN values

In [ ]:
from tensorflow.keras.datasets import mnist
from sklearn.model_selection import StratifiedKFold
import numpy as np

# Load dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Check for null (NaNs) in dataset
if np.isnan(x_train).any() or np.isnan(x_test).any():
    print("Προσοχή: Βρέθηκαν κενές (NaN) τιμές στο dataset!")
else:
    print("Δεν βρέθηκαν κενές (NaN) τιμές στο dataset. Συνεχίζεται η επεξεργασία.")

# Flattening
x_train_flat = x_train.reshape(60000, -1)

# Create StratifiedKFold w/ 6 folds
skf = StratifiedKFold(n_splits=6, shuffle=True, random_state=42)

# List to store datasets
subsets = []

print(f'Συνολικό μέγεθος train set: {len(x_train)}\n')

for i, (train_index, test_index) in enumerate(skf.split(x_train_flat, y_train)):
    # Keep test_index for every fold to take back 6 sets of 10.000 records
    x_subset = x_train[test_index]
    y_subset = y_train[test_index]
    subsets.append((x_subset, y_subset))
    print(f'Dataset {i+1}: Shape {x_subset.shape}, Labels: {np.unique(y_subset, return_counts=True)[1]}')

# Validate size
print(f'\nΔημιουργήθηκαν {len(subsets)} υποσύνολα.')

Εκπαίδευση του μοντέλου

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

# Preprocess the data
x_train_scaled = x_train.reshape((60000, 28, 28, 1)).astype('float32') / 255
x_test_scaled = x_test.reshape((10000, 28, 28, 1)).astype('float32') / 255

# Create an independent validation set from the training data
# We take 20% of the training data for validation
x_train_final, x_val, y_train_final, y_val = train_test_split(
    x_train_scaled, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Define the architecture according to requirements
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train the model with the independent validation set
history = model.fit(x_train_final, y_train_final, epochs=5,
                    validation_data=(x_val, y_val))

# Print final losses
final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]

print(f'\nFinal Training Loss: {final_train_loss:.4f}')
print(f'Final Validation Loss: {final_val_loss:.4f}')

Training & Validation Curves

In [ ]:
import matplotlib.pyplot as plt

# Create the plot
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss', marker='o')
plt.plot(history.history['val_loss'], label='Validation Loss', marker='o')

# Add details to the plot
plt.title('Training and Validation Loss Curves')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.xticks(range(len(history.history['loss'])))
plt.show()

Performance on Training Subset

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# 1. Prepare the data for prediction
# Using the first subset from the 6 we created (subsets[0])
x_sub_train, y_sub_train = subsets[0]
x_sub_train_scaled = x_sub_train.reshape((-1, 28, 28, 1)).astype('float32') / 255

# 2. Make predictions
train_preds_probs = model.predict(x_sub_train_scaled)
test_preds_probs = model.predict(x_test_scaled)

# 3. Convert probabilities to class labels (argmax)
train_preds = np.argmax(train_preds_probs, axis=1)
test_preds = np.argmax(test_preds_probs, axis=1)

# 4. Calculate and print performance metrics
print("--- Performance on Training Subset (10,000 samples) ---")
print(f"Accuracy: {accuracy_score(y_sub_train, train_preds):.4f}")
print(classification_report(y_sub_train, train_preds))

print("\n--- Performance on Test Set (10,000 samples) ---")
print(f"Accuracy: {accuracy_score(y_test, test_preds):.4f}")
print(classification_report(y_test, test_preds))

Πειράματα για DNN, CNN & Scores

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras import layers, models
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

results_data = []

def get_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
    return acc, prec, rec, f1

def build_dnn():
    model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn():
    model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

x_test_reshaped = x_test.reshape((-1, 28, 28, 1)).astype('float32') / 255

print('Έναρξη πειραμάτων...')
for tech_name, model_builder in [('DNN', build_dnn), ('CNN', build_cnn)]:
    for fold_idx, (x_sub, y_sub) in enumerate(subsets):
        x_sub_scaled = x_sub.reshape((-1, 28, 28, 1)).astype('float32') / 255
        model = model_builder()
        model.fit(x_sub_scaled, y_sub, epochs=3, verbose=0)

        for set_label, x_eval, y_eval in [('Train', x_sub_scaled, y_sub), ('Test', x_test_reshaped, y_test)]:
            preds = np.argmax(model.predict(x_eval, verbose=0), axis=1)
            acc, prec, rec, f1 = get_metrics(y_eval, preds)
            results_data.append({
                'Technique name': tech_name,
                'Set': set_label,
                'Fold number': fold_idx + 1,
                'Accuracy': acc,
                'Precision': prec,
                'Recall': rec,
                'F1 score': f1
            })
    print(f'Ολοκληρώθηκαν τα πειράματα για: {tech_name}')

df_results = pd.DataFrame(results_data)
df_results.to_csv('erotima1.csv', index=False)
print('\nΤο αρχείο erotima1.csv δημιουργήθηκε με επιτυχία.')
display(df_results)

Επιόγή καλύτερων Folds για DNN, CNN & αποθήκευση μοντέλων

In [ ]:
# Find the best fold for DNN
best_dnn_row = df_results[(df_results['Technique name'] == 'DNN') & (df_results['Set'] == 'Test')].sort_values(by='Accuracy', ascending=False).iloc[0]
best_dnn_fold = int(best_dnn_row['Fold number'])

# Find the best fold for CNN
best_cnn_row = df_results[(df_results['Technique name'] == 'CNN') & (df_results['Set'] == 'Test')].sort_values(by='Accuracy', ascending=False).iloc[0]
best_cnn_fold = int(best_cnn_row['Fold number'])

print(f'Best DNN Fold: {best_dnn_fold} (Accuracy: {best_dnn_row["Accuracy"]:.4f})')
print(f'Best CNN Fold: {best_cnn_fold} (Accuracy: {best_cnn_row["Accuracy"]:.4f})')

# Re-train and save the best DNN
x_sub, y_sub = subsets[best_dnn_fold - 1]
x_sub_scaled = x_sub.reshape((-1, 28, 28, 1)).astype('float32') / 255
dnn_model = build_dnn()
dnn_model.fit(x_sub_scaled, y_sub, epochs=3, verbose=0)
dnn_model.save('best_dnn_model.h5')

# Re-train and save the best CNN
x_sub, y_sub = subsets[best_cnn_fold - 1]
x_sub_scaled = x_sub.reshape((-1, 28, 28, 1)).astype('float32') / 255
cnn_model = build_cnn()
cnn_model.fit(x_sub_scaled, y_sub, epochs=3, verbose=0)
cnn_model.save('best_cnn_model.h5')

print('\nΤα μοντέλα best_dnn_model.h5 και best_cnn_model.h5 αποθηκεύτηκαν επιτυχώς.')

Confusion Matrices & ROC Curves για best DNN & CNN

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from tensorflow.keras.models import load_model

def plot_performance(model_path, title_suffix, x_data, y_true):
    # 1. Load model and predict
    model = load_model(model_path)
    y_pred_probs = model.predict(x_data)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # 2. Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {title_suffix}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

    # 3. ROC Curve
    y_true_bin = label_binarize(y_true, classes=range(10))
    n_classes = 10
    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    plt.figure(figsize=(8, 6))
    colors = plt.cm.get_cmap('tab10')(np.linspace(0, 1, n_classes))
    for i, color in zip(range(n_classes), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2,
                 label=f'Class {i} (AUC = {roc_auc[i]:0.2f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curves - {title_suffix}')
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()

# Prepare test data
x_test_reshaped = x_test.reshape((-1, 28, 28, 1)).astype('float32') / 255

print("--- Evaluating DNN Model ---")
plot_performance('best_dnn_model.h5', 'Best DNN', x_test_reshaped, y_test)

print("\n--- Evaluating CNN Model ---")
plot_performance('best_cnn_model.h5', 'Best CNN', x_test_reshaped, y_test)